# NB 06 — Memory primitives (✦ spine touchpoint)

**Goal:** introduce three memory shapes via `agentlab.memory`:
`ConversationBuffer`, `KeyValueMemory`, `SemanticMemory`. Then layer
`KeyValueMemory` into the research-assistant spine so it can answer
follow-up questions.

Heads up: NB 07 will depend on `SemanticMemory`. NB 08 will trace
everything.

In [1]:
from _common import chdir_to_repo_root, cost_banner, load_env

load_env()
chdir_to_repo_root()
cost_banner(
    notebook="06 — Memory primitives (✦ spine + memory)",
    estimate="$0.03",
    model="claude-sonnet-4-6",
)

╭────────────── Notebook info ──────────────╮
│ 06 — Memory primitives (✦ spine + memory) │
│ Default model: claude-sonnet-4-6          │
│ Estimated cost per full run: $0.03        │
╰───────────────────────────────────────────╯

## Step 1 — Short-term: `ConversationBuffer`

Token-bounded message buffer with optional summarization. The simplest
memory: just keep the last N tokens of conversation.

In [2]:
from agentlab.llm import DEFAULT_MODEL, get_client
from agentlab.memory import ConversationBuffer

client = get_client()

buf = ConversationBuffer(max_tokens=80)  # tiny on purpose: forces truncation
for i in range(8):
    buf.append({"role": "user", "content": f"This is message {i}, which has some words in it for token weight."})
    buf.append({"role": "assistant", "content": f"Acknowledged message {i}."})

print(f"All messages: {len(buf.messages())}")
truncated = buf.truncate()
dropped = len(buf.messages()) - len(truncated)
print(f"After truncate: {len(truncated)} ({dropped} oldest dropped to fit budget)")
print("Oldest kept:", truncated[0]["content"][:60])
print("Newest kept:", truncated[-1]["content"][:60])

All messages: 16
After truncate: 8 (8 oldest dropped to fit budget)
Oldest kept: This is message 4, which has some words in it for token weig
Newest kept: Acknowledged message 7.


### Summarization keeps context at a price

`truncate()` *drops* old messages. `summarize()` *compresses* them
into a single summary string. Use it when the conversation has
load-bearing context older than your budget.

In [3]:
summary = buf.summarize(client)
print("Summary:", summary)

Summary: **Conversation Summary:**

The user sent eight sequential messages (numbered 0-7), each with identical structure containing the phrase "This is message [number], which has some words in it for token weight." I acknowledged receipt of each message in turn. No decisions were made or key facts exchanged beyond the simple message confirmations.


## Step 2 — Long-term: `KeyValueMemory`

Persistent dict. Use cases: user preferences, learned facts,
pre-computed results. `save / load` round-trip via JSON.

In [4]:
from pathlib import Path

from agentlab.memory import KeyValueMemory

kv = KeyValueMemory()
kv.set("user_name", "Rob")
kv.set("project_focus", "AI agents curriculum")
kv.set("last_question", "How do I build NB 06?")

scratch = Path(".kv-demo.json")
kv.save(scratch)
print(scratch.read_text())

fresh = KeyValueMemory()
fresh.load(scratch)
assert fresh.get("user_name") == "Rob"
print(f"\nRe-loaded: user_name = {fresh.get('user_name')}")

scratch.unlink()

{
  "user_name": "Rob",
  "project_focus": "AI agents curriculum",
  "last_question": "How do I build NB 06?"
}

Re-loaded: user_name = Rob


## Step 3 — Semantic recall: `SemanticMemory`

Embedding-backed retrieval. The first run downloads the
sentence-transformers model (~80 MB). Subsequent runs are fast.

In [5]:
from agentlab.memory import SemanticMemory

sm = SemanticMemory(collection_name="nb06_demo")
sm.add("Anthropic's Claude API uses an HTTP-based message protocol.", metadata={"topic": "api"})
sm.add("MCP (Model Context Protocol) standardizes how tools are exposed to LLMs.", metadata={"topic": "mcp"})
sm.add("ReAct interleaves reasoning and acting in a loop.", metadata={"topic": "react"})
sm.add("Pydantic validates structured outputs from LLMs.", metadata={"topic": "structured"})

for query in ["How does the Anthropic API work?", "What is the protocol for tool exposure?"]:
    matches = sm.query(query, top_k=2)
    print(f"\nQuery: {query}")
    for m in matches:
        print(f"  [{m.score:.3f}] {m.text}  (topic={m.metadata.get('topic')})")

/home/rob/PythonEnvironments/Agents/.agents/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17271.55it/s]



Query: How does the Anthropic API work?
  [0.718] Anthropic's Claude API uses an HTTP-based message protocol.  (topic=api)
  [0.211] MCP (Model Context Protocol) standardizes how tools are exposed to LLMs.  (topic=mcp)

Query: What is the protocol for tool exposure?
  [0.451] MCP (Model Context Protocol) standardizes how tools are exposed to LLMs.  (topic=mcp)
  [0.220] Anthropic's Claude API uses an HTTP-based message protocol.  (topic=api)


## Step 4 — Spine: research assistant with memory

We redefine `research()` from NB 03 inline (per the curriculum's
"no library extraction until NB 15" rule), but this time the
assistant has a `KeyValueMemory` it can read on each call. We expose
two extra tools: `remember(key, value)` and `recall(key)`.

In [6]:
from agentlab.tools import ToolRegistry
from agentlab.types import Answer, Citation

registry = ToolRegistry()
session_memory = KeyValueMemory()


@registry.tool(description="Save a fact for later retrieval. Use when the user mentions a preference or fact.")
def remember(key: str, value: str) -> str:
    session_memory.set(key, value)
    return f"Saved {key}={value}"


@registry.tool(description="Recall a previously-saved fact by key. Returns the stored value or 'not found'.")
def recall(key: str) -> str:
    val = session_memory.get(key)
    return str(val) if val is not None else "not found"


submit_answer_tool = {
    "name": "submit_answer",
    "description": "Submit your final answer with citations.",
    "input_schema": Answer.model_json_schema(),
}

SPINE_SYSTEM = """You are a research assistant with persistent memory.

Workflow (do these in order, every turn):
1. If the user mentions a preference, project, or fact about themselves, call remember(key, value) to save it.
2. If a follow-up question references something earlier, call recall(key) to look it up.
3. ALWAYS call web_search at least once before answering — even for recommendations or
   follow-ups, you need fresh sources to cite. Do not skip this step.
4. Finish by calling submit_answer exactly once with:
   - summary: 2-4 sentences answering the question (weave in any recalled facts).
   - citations: list of {url, title, optional quote} drawn from your web_search results.
     At least one entry; ideally 2-3. URLs must come from web_search — do not invent any.
5. Do not produce free-form text instead of submit_answer — your final output must be a submit_answer call."""


def research_with_memory(question: str, max_turns: int = 6) -> Answer:
    # Surface what's in memory so the agent doesn't have to guess key names.
    keys = session_memory.keys()
    system = SPINE_SYSTEM + (
        f"\n\nCurrent memory keys (call recall(key) to read each value): {keys}"
        if keys
        else "\n\nMemory is currently empty."
    )
    messages = [{"role": "user", "content": question}]
    tools = registry.schemas() + [
        {"type": "web_search_20250305", "name": "web_search"},
        submit_answer_tool,
    ]
    handlers = registry.handlers()
    for turn in range(max_turns):
        response = client.messages.create(
            model=DEFAULT_MODEL, max_tokens=2048, system=system,
            tools=tools, messages=messages,
        )
        block_descs = [
            f"{getattr(b, 'type', '?')}"
            + (f":{b.name}" if getattr(b, "name", None) else "")
            for b in response.content
        ]
        print(
            f"  [debug turn {turn}] stop={response.stop_reason} "
            f"out_tokens={response.usage.output_tokens} blocks={block_descs}"
        )
        for block in response.content:
            if block.type == "tool_use" and block.name == "submit_answer":
                return Answer.model_validate(block.input)
        # Run any local tool calls; web_search is managed.
        tool_results = []
        for block in response.content:
            if block.type == "tool_use" and block.name in handlers:
                try:
                    out = handlers[block.name](**block.input)
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(out)})
                except Exception as e:
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(e), "is_error": True})
        messages.append({"role": "assistant", "content": response.content})
        if tool_results:
            messages.append({"role": "user", "content": tool_results})
        if response.stop_reason == "end_turn":
            raise RuntimeError("end_turn without submit_answer.")
    raise RuntimeError("Out of turns.")


# Turn 1: introduce a preference.
answer1 = research_with_memory(
    "I'm working on a Python async-first project. Remember that. "
    "What's the most popular async HTTP library?"
)
print("Q1 →", answer1.summary[:200])
print(f"\n[memory now contains: {session_memory.keys()}]")

# Turn 2: follow-up that needs the remembered context.
answer2 = research_with_memory(
    "Given my project's focus, which library do you recommend I use? "
    "(Look up what you remembered.)"
)
print("\nQ2 →", answer2.summary[:300])

  [debug turn 0] stop=tool_use out_tokens=119 blocks=['tool_use:remember', 'server_tool_use:web_search']
  [debug turn 1] stop=tool_use out_tokens=586 blocks=['web_search_tool_result', 'tool_use:submit_answer']
Q1 → For your Python async-first project, the two most popular async HTTP libraries are **aiohttp** and **HTTPX**. aiohttp is the go-to choice for pure async work — <cite index="5-6,5-7">it's a powerful as

[memory now contains: ['project_type']]
  [debug turn 0] stop=tool_use out_tokens=54 blocks=['tool_use:recall']
  [debug turn 1] stop=tool_use out_tokens=858 blocks=['server_tool_use:web_search', 'web_search_tool_result', 'tool_use:submit_answer']

Q2 → Since your project is an **async-first Python project**, I recommend **FastAPI** as your primary framework/library, with **asyncio** as the backbone and **uvloop** for a performance boost. Here's why:

- **FastAPI** is the top recommendation for async Python in 2025. <cite index="5-7,5-8">It is a mo


## Reflect

- **Three memory shapes, three jobs.** Conversation buffers handle
  token budgets; key-value stores hold facts; semantic memory finds
  things by meaning.
- **Memory becomes a tool the agent uses.** `remember/recall` are
  plain `ToolRegistry` tools — the agent decides when to use them.
- **Why `KeyValueMemory` for the spine, not `ConversationBuffer`?**
  Because the spine is multi-turn but stateless across calls — we
  want explicit, named facts (preferences, project context), not
  raw transcripts.
- **Foreshadow:** NB 07 puts `SemanticMemory` to work — but exposed as
  a *retrieval tool*, not unconditional context injection.